In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.tar.gz
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/example_submission.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv
/kaggle/input/notebooks/sandrolomidze42/model-py/__results__.html
/kaggle/input/notebooks/sandrolomidze42/model-py/__notebook__.ipynb
/kaggle/input/notebooks/sandrolomidze42/model-py/__output__.json
/kaggle/input/notebooks/sandrolomidze42/model-py/custom.css


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense

def create_simple_cnn():
    """
    Minimal CNN: 1 Conv layer + ReLU -> Flatten -> Dense Output
    """
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(48, 48, 1)),
        Flatten(),
        Dense(7, activation='softmax')
    ])
    return model

In [4]:


model = create_simple_cnn()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1781625641.894011      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781625641.900192      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 67712)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │       473,991 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 474,311 (1.81 MB)

 Trainable params: 474,311 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load training data
train_df = pd.read_csv('/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv')

# Convert pixel strings to arrays and reshape to 48x48 images
pixels = train_df['pixels'].apply(lambda x: np.array(x.split(), dtype=np.float32))
images = np.stack(pixels).reshape(-1, 48, 48, 1)

# Normalize pixel values to 
images = images / 255.0

# Labels (0-6)
labels = train_df['emotion'].values

# Split into training and validation sets (80% train, 20% val)
X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")


Training set shape: (22967, 48, 48, 1)
Validation set shape: (5742, 48, 48, 1)


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# Fit the generator to the training data
datagen.fit(X_train)


In [8]:
import tensorflow as tf

# Define early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# Train the model
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)


Epoch 1/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.3981 - loss: 1.5741 - val_accuracy: 0.4404 - val_loss: 1.4915
Epoch 2/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.3994 - loss: 1.5665 - val_accuracy: 0.4413 - val_loss: 1.4805
Epoch 3/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4005 - loss: 1.5639 - val_accuracy: 0.4410 - val_loss: 1.4915
Epoch 4/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4102 - loss: 1.5581 - val_accuracy: 0.4375 - val_loss: 1.4908
Epoch 5/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4060 - loss: 1.5588 - val_accuracy: 0.4462 - val_loss: 1.4849
Epoch 6/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4044 - loss: 1.5598 - val_accuracy: 0.4316 - val_loss: 1.5070
Epoch 7/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4073 - loss: 1.5561 - val_accuracy: 0.4422 - val_loss: 1.4781
Epoch 8/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4037 - loss: 1.5546 - val_accu

In [9]:
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")


180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4464 - loss: 1.4623
Validation Loss: 1.4623
Validation Accuracy: 0.4464
